# Notebook 4 – Hyperparameter Tuning

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.

## Setup: Load & Split Data

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. What is Hyperparameter Tuning?
The process of trying different hyperparameter values to find the combination that gives the **best performance**, usually measured with cross validation.

In [2]:
m = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
print("One manual guess — Test acc:", accuracy_score(y_test, m.predict(X_test)))

One manual guess — Test acc: 0.8933333333333333


## 2. Manual Tuning
Trying values **by hand**, one at a time, based on intuition or trial and error. Simple but slow and easy to miss good combinations.

In [3]:
for depth in [2, 4, 6, 8]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    print(f"max_depth={depth}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

max_depth=2: Test acc=0.897
max_depth=4: Test acc=0.893
max_depth=6: Test acc=0.890
max_depth=8: Test acc=0.882


## 3. Grid Search
Tries **every combination** of specified hyperparameter values exhaustively. Guaranteed to find the best combination within the grid, but can be slow if the grid is large.

In [5]:
from sklearn.model_selection import GridSearchCV
param_grid = {'max_depth': [2, 4, 6, None], 'min_samples_split': [2, 20, 50]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid.fit(X_train, y_train)
print("Combinations tried:", len(grid.cv_results_['params']))

Combinations tried: 12


## 4. Random Search
Tries a **random sample** of combinations instead of all of them. Faster than Grid Search, and often finds a nearly-as-good result with far fewer trials — especially useful with large search spaces.

In [6]:
from sklearn.model_selection import RandomizedSearchCV
param_dist = {'max_depth': [2, 4, 6, 8, None], 'min_samples_split': [2, 10, 20, 50, 100]}
random_search = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), param_dist, n_iter=6, cv=5, random_state=42)
random_search.fit(X_train, y_train)
print("Combinations tried:", len(random_search.cv_results_['params']))

Combinations tried: 6


## 5. Cross Validation
Tuning uses **cross validation** internally — each candidate combination is scored across multiple folds, not just one split, so the chosen hyperparameters generalize better.

In [7]:
print("Each candidate's CV scores across 5 folds (Grid Search, first candidate):")
print(grid.cv_results_['mean_test_score'][:5])

Each candidate's CV scores across 5 folds (Grid Search, first candidate):
[0.89666667 0.89666667 0.89666667 0.89333333 0.895     ]


## 6. GridSearchCV
The scikit-learn tool that performs Grid Search **combined with** cross validation automatically — we already used it above.

In [8]:
print("Grid Search best params:", grid.best_params_)

Grid Search best params: {'max_depth': 2, 'min_samples_split': 2}


## 7. RandomizedSearchCV
The scikit-learn tool that performs Random Search **combined with** cross validation automatically — we already used it above.

In [9]:
print("Random Search best params:", random_search.best_params_)

Random Search best params: {'min_samples_split': 2, 'max_depth': 2}


## 8. Search Space
The set of hyperparameter values considered during tuning (e.g., `max_depth: [2, 4, 6, None]`). A well-chosen search space is critical — too narrow may miss the best model, too wide wastes time.

In [10]:
print("Grid search space:", param_grid)
print("Random search space:", param_dist)

Grid search space: {'max_depth': [2, 4, 6, None], 'min_samples_split': [2, 20, 50]}
Random search space: {'max_depth': [2, 4, 6, 8, None], 'min_samples_split': [2, 10, 20, 50, 100]}


## 9. Scoring Parameter
Tells the search **what metric to optimize for** (accuracy, F1, precision, etc). Different goals need different scoring — e.g. use `'f1'` if false negatives matter more than raw accuracy.

In [11]:
grid_f1 = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='f1')
grid_f1.fit(X_train, y_train)
print("Best params optimizing for F1:", grid_f1.best_params_)

Best params optimizing for F1: {'max_depth': 2, 'min_samples_split': 2}


## 10. Best Parameters
The specific hyperparameter combination that scored highest during the search — this is what you'd use for your final model.

In [12]:
print("Grid Search best_params_:", grid.best_params_)
print("Random Search best_params_:", random_search.best_params_)

Grid Search best_params_: {'max_depth': 2, 'min_samples_split': 2}
Random Search best_params_: {'min_samples_split': 2, 'max_depth': 2}


## 11. Best Score
The cross-validated score achieved by the best parameter combination — this is our estimate of how well the tuned model will perform.

In [13]:
print("Grid Search best_score_:", grid.best_score_)
print("Random Search best_score_:", random_search.best_score_)

Grid Search best_score_: 0.8966666666666667
Random Search best_score_: 0.8966666666666667
